In [1]:
%pip install transformers[ja,sentencepiece,torch]

In [2]:
from transformers import pipeline

In [3]:
text_classification_pipeline = pipeline(
    model = "llm-book/bert-base-japanese-v3-marc_ja"
)
positive_text = "世界には言葉がわからなくても感動する音楽がある"
print(text_classification_pipeline(positive_text)[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: llm-book/bert-base-japanese-v3-marc_ja
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

{'label': 'positive', 'score': 0.9994138479232788}


In [4]:
print(text_classification_pipeline("ほなすけなんか喉痛い気がする。やばそう。")[0])

{'label': 'negative', 'score': 0.6954834461212158}


In [5]:
# NLI: natural Language Inference(自然言語推論)
nli_pipeline = pipeline(
    model = "llm-book/bert-base-japanese-v3-jnli"
)
text = "二人の男性がジェット機を見ています"
entailment_text = "ジェット機を見ている二人がいます"  # entailment: 含意

# textとentailment_textの論理関係を推測
print(nli_pipeline({"text": text, "text_pair": entailment_text}))


config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: llm-book/bert-base-japanese-v3-jnli
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

{'label': 'entailment', 'score': 0.9964165687561035}


In [6]:
contradiction_text = "二人の男性が飛んでいます"  # 矛盾
print(nli_pipeline({"text": text, "text_pair": contradiction_text}))


{'label': 'contradiction', 'score': 0.9922866821289062}


In [7]:
neutral_text = "2人の男性が白い飛行機を眺めています"
print(nli_pipeline({"text": text, "text_pair": neutral_text}))
# 2人を二人に変えると"contradiction"になる

{'label': 'neutral', 'score': 0.950067400932312}


In [8]:
# STS: semantic textual similarity (意味的類推度計算)
text_sim_pipeline = pipeline(
    model = "llm-book/bert-base-japanese-v3-jsts"
)
text = "川べりでサーフボードを持った人たちがいます"
sim_text = "サーファーたちが川べりに立っています"
result = text_sim_pipeline({"text": text, "text_pair": sim_text})
print(result["score"])

config.json:   0%|          | 0.00/796 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: llm-book/bert-base-japanese-v3-jsts
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

2.152515172958374


In [9]:
dissim_text = "トイレの壁に黒いタオルが掛けられています"
result = text_sim_pipeline({"text": text, "text_pair": dissim_text})
print(result["score"])

0.8160146474838257


In [12]:
from torch.nn.functional import cosine_similarity

sim_enc_pipeline = pipeline(
    model = "llm-book/bert-base-japanese-v3-unsup-simcse-jawiki",
    task = "feature-extraction",
)

# text と sim_textのベクトルを獲得
text_emb = sim_enc_pipeline(text, return_tensors=True)[0][0]
sim_emb = sim_enc_pipeline(sim_text, return_tensors=True)[0][0]
# print(f"text_emb = {text_emb}")
# print(f"sim_emb = {sim_emb}")

# text と sim_textの類似度を計算
sim_pair_score = cosine_similarity(text_emb, sim_emb, dim=0)
print(sim_pair_score.item())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: llm-book/bert-base-japanese-v3-unsup-simcse-jawiki
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.8568588495254517


In [14]:
# dissim_textとのベクトルを獲得
dissim_emb = sim_enc_pipeline(dissim_text, return_tensors=True)[0][0]
dissim_pair_score = cosine_similarity(text_emb, dissim_emb, dim=0)
print(dissim_pair_score.item())

0.45993196964263916


In [ ]:
# NER: Named entity recognition (固有表現認識)
from pprint import pprint

ner_pipeline = pipeline(
    model = "llm-book/bert-base-japanese-v3-ner-wikipedia-dataset",
    aggregation_strategy = "simple"
)
text = "大谷翔平は岩手県水沢市出身のプロ野球選手"
pprint(ner_pipeline(text))